# Ativação de conta: modelo de propensão + MAB contextual com braço nulo

**Problema (ver `contexto.md`).** Clientes abrem a conta e não ativam. Ativar = fazer
*pix*, *pagamento*, *seguro* ou *investimento* (existem outras formas, que ficam para depois).
Um agente conversacional vai falar com o cliente; queremos **controlar o assunto** da conversa
(o agente escolhe o tom, nós escolhemos o tema).

**Arquitetura da solução deste notebook**

1. **Modelo de propensão** `p0 = P(ativar em 7 dias | features, sem intervenção)`.
   Treinado no histórico, que é 100% sem agente. Ele responde: *"esse cliente ativaria sozinho?"*
2. **MAB contextual** escolhe o **assunto da mensagem**. O contexto inclui `p0` como feature.
   O bandit aprende o **uplift** (efeito incremental) de cada assunto em função de `p0` e de
   algumas features cruas.
3. **Braço nulo** (`sem_mensagem`) é um dos braços. É ele que permite identificar os clientes
   em que **o agente atrapalha mais do que ajuda** (os *sleeping dogs*): se o braço nulo é o
   argmax do cliente, a política simplesmente não fala com ele.

**Por que propensão como feature (e não só as features cruas)?**
`p0` é um resumo unidimensional e calibrado de "quão perto do sim esse cliente está".
O uplift tipicamente é uma função em forma de sino de `p0`: quem ativaria de qualquer jeito
(`p0` alto) não tem o que ganhar — e pode se irritar; quem não ativaria nunca (`p0` baixo)
não reage. O ganho está no meio. Dar `p0` e `p0²` ao bandit deixa ele aprender essa curva com
muito menos dados do que se tivesse que redescobri-la a partir das features originais.

**Pontos de produção que o notebook respeita de propósito**

- **Janela de atribuição de 7 dias** ⇒ recompensa atrasada ⇒ o bandit atualiza em **lote**
  (aqui, semanal), não a cada evento.
- **Grupo controle de 20%** que nunca entra no fluxo, para medir o ganho do programa inteiro.
- **Cold start**: ninguém recebeu recomendação ainda, então o bandit começa do zero e a
  exploração é responsabilidade do Thompson Sampling.
- **Estado do modelo serializável** (JSON) para servir e atualizar em produção.

In [ ]:
# ---------------------------------------------------------------------------------------------
# Imports e configuração global
# ---------------------------------------------------------------------------------------------
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.calibration import calibration_curve

SEED = 42
rng_global = np.random.default_rng(SEED)

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)

# Números do negócio (contexto.md) --------------------------------------------------------------
CLIENTES_POR_SEMANA = 29_000
TAXA_ATIVACAO_D7 = 0.18      # alvo do gerador de dados sintéticos
TAXA_ATIVACAO_D0 = 0.05      # ou seja, ~28% dos ativadores ativam no mesmo dia
JANELA_ATRIBUICAO_DIAS = 7   # define o atraso da recompensa
FRACAO_CONTROLE = 0.20       # 20% nunca entram no fluxo do agente

# Mix de canal de ativação entre quem ativa (o "restante" é 'outros')
MIX_CANAL_ATIVACAO = {
    "pix": 0.357,
    "pagamento": 0.2118,
    "seguro": 0.203,
    "investimento": 0.062,
    "outros": 1 - (0.357 + 0.2118 + 0.203 + 0.062),
}

# Braços do bandit = assuntos que o agente pode puxar. O braço 0 é o braço NULO.
BRACOS = ["sem_mensagem", "msg_pix", "msg_pagamento", "msg_seguro", "msg_investimento"]
BRACO_NULO = 0
# Mapeia cada braço para o canal de ativação que ele tenta empurrar (o nulo não empurra nada).
CANAL_DO_BRACO = {1: "pix", 2: "pagamento", 3: "seguro", 4: "investimento"}

print(f"{len(BRACOS)} braços: {BRACOS}")
print(f"Mix de canal alvo: { {k: round(v, 4) for k, v in MIX_CANAL_ATIVACAO.items()} }")

## 1. Dados sintéticos

O gerador é o "mundo real" simulado. **Em produção ele não existe** — serve para (a) treinar o
modelo de propensão como se fosse o histórico e (b) responder ao bandit durante a simulação,
funcionando como o cliente que converte ou não.

### Como o mundo é construído

Para cada cliente sorteamos features observáveis no momento da abertura da conta e três coisas
latentes (que o modelo **não** vê diretamente, só através de proxies):

| Latente | Papel |
|---|---|
| `p0` | probabilidade de ativar em 7 dias **sem** intervenção |
| `afinidade[canal]` | qual assunto interessa esse cliente (softmax sobre pix/pagamento/seguro/investimento/outros) |
| `sensivel_msg` | cliente que se **incomoda** com a abordagem (o que gera os *sleeping dogs*) |

A probabilidade sob o braço `a` é montada em espaço de probabilidade, para ficar legível:

```
p[a] = clip( p0  +  GANHO_MAX * afinidade[canal(a)] * persuadabilidade(p0)  -  dano ,  0, 1 )

persuadabilidade(p0) = exp(-0.5 * ((p0 - 0.15) / 0.13)²)   # sino com pico em p0 = 0.15
dano = DANO_MAX * sensivel_msg * (0.3 + 2.0 * p0)          # irritar quem já ia ativar custa mais
```

A `persuadabilidade` é o coração da história: quem já ia ativar (`p0` alto) não tem o que
ganhar, e ao mesmo tempo é quem mais perde ao ser incomodado — o `dano` cresce com `p0`.
É exatamente aí que o braço nulo tem que vencer.

Três consequências importantes, e todas são o ponto do exercício:

- o **melhor assunto varia por cliente** (via `afinidade`) → o bandit precisa ser contextual;
- o **tamanho do efeito varia com `p0`** → daí `p0` e `p0²` no contexto;
- para parte dos clientes **todo braço com mensagem é pior que o nulo** → o braço nulo tem
  trabalho a fazer. Note que `sensivel_msg` **não** é uma feature do modelo: ele só consegue
  identificar esses clientes por correlação com idade, canal de aquisição e engajamento.

In [ ]:
# ---------------------------------------------------------------------------------------------
# Parâmetros do "mundo" (ground truth). Só o simulador conhece isso.
# ---------------------------------------------------------------------------------------------
GANHO_MAX = 0.35    # escala do ganho de um assunto perfeitamente alinhado ao cliente
DANO_MAX = 0.22     # escala do dano em quem é sensível a ser abordado
PICO_PERSUASAO = 0.15   # propensão em que a mensagem tem efeito máximo
LARGURA_PERSUASAO = 0.13


def _z(x):
    """Padroniza (z-score). Usado só dentro do gerador."""
    x = np.asarray(x, dtype=float)
    return (x - x.mean()) / (x.std() + 1e-9)


def _softmax(logits):
    logits = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(logits)
    return e / e.sum(axis=1, keepdims=True)


def _sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


CANAIS = ["pix", "pagamento", "seguro", "investimento", "outros"]


def gerar_populacao(n, rng, intercepto_p0=None, intercepto_canal=None):
    """Sorteia `n` clientes que acabaram de abrir a conta.

    Devolve (df_features_observaveis, dict_latentes). Os interceptos podem ser passados para
    reutilizar a calibração feita uma única vez (ver célula seguinte) — assim toda coorte nova
    sai com a mesma taxa de ativação base e o mesmo mix de canal.
    """
    # --- features observáveis no momento da abertura ------------------------------------------
    idade = rng.normal(34, 10, n).clip(18, 75)
    renda = np.exp(rng.normal(np.log(2500), 0.6, n)).clip(800, 40_000)
    score_bureau = rng.normal(600, 120, n).clip(200, 1000)
    canal_aquisicao = rng.choice(["organico", "pago", "indicacao"], n, p=[0.45, 0.40, 0.15])
    ja_tem_conta_outro_banco = rng.binomial(1, 0.70, n)
    sessoes_app_d0 = rng.poisson(2.0, n)
    cadastro_completo = rng.binomial(1, 0.80, n)
    dispositivo_ios = rng.binomial(1, 0.35, n)
    cartao_aprovado = rng.binomial(1, 0.55, n)

    df = pd.DataFrame({
        "idade": idade,
        "renda_declarada": renda,
        "score_bureau": score_bureau,
        "canal_aquisicao": canal_aquisicao,
        "ja_tem_conta_outro_banco": ja_tem_conta_outro_banco,
        "sessoes_app_d0": sessoes_app_d0,
        "cadastro_completo": cadastro_completo,
        "dispositivo_ios": dispositivo_ios,
        "cartao_aprovado": cartao_aprovado,
    })

    log_renda = np.log(renda)
    pago = (canal_aquisicao == "pago").astype(float)
    indicacao = (canal_aquisicao == "indicacao").astype(float)
    organico = (canal_aquisicao == "organico").astype(float)

    # --- latente 1: propensão base p0 (sem nenhuma intervenção) -------------------------------
    # Engajamento no app e cadastro completo puxam ativação para cima; tráfego pago para baixo.
    # O ruído gaussiano é a heterogeneidade que nenhum modelo consegue explicar.
    z_p0 = (
        0.55 * _z(sessoes_app_d0)
        + 0.35 * cadastro_completo
        + 0.30 * cartao_aprovado
        + 0.25 * indicacao
        - 0.20 * pago
        + 0.20 * _z(log_renda)
        + 0.15 * _z(score_bureau)
        - 0.10 * _z(idade)
        + rng.normal(0, 0.40, n)
    )
    if intercepto_p0 is None:
        # Calibra o intercepto por bisseção para a média de p0 bater TAXA_ATIVACAO_D7.
        lo, hi = -8.0, 4.0
        for _ in range(80):
            mid = (lo + hi) / 2
            if _sigmoid(z_p0 + mid).mean() < TAXA_ATIVACAO_D7:
                lo = mid
            else:
                hi = mid
        intercepto_p0 = (lo + hi) / 2
    p0 = _sigmoid(z_p0 + intercepto_p0)

    # --- latente 2: afinidade por assunto -----------------------------------------------------
    # Quem já tem conta em outro banco tende a ativar por pix; mais velhos por seguro;
    # renda alta por investimento; renda baixa por pagamento de boleto.
    logits = np.column_stack([
        0.50 * ja_tem_conta_outro_banco + 0.30 * _z(sessoes_app_d0),        # pix
        0.40 * (-_z(log_renda)) + 0.20 * cadastro_completo,                 # pagamento
        0.50 * _z(idade) + 0.20 * organico,                                 # seguro
        0.90 * _z(log_renda) + 0.30 * _z(score_bureau),                     # investimento
        np.zeros(n),                                                        # outros (referência)
    ])
    if intercepto_canal is None:
        # Ajusta os interceptos para o mix observado ENTRE OS ATIVADORES bater o mix do negócio.
        alvo = np.array([MIX_CANAL_ATIVACAO[c] for c in CANAIS])
        intercepto_canal = np.zeros(5)
        ativou_aprox = rng.random(n) < p0                     # amostra de ativadores
        for _ in range(300):
            probs = _softmax(logits + intercepto_canal)
            obs = probs[ativou_aprox].mean(axis=0)
            intercepto_canal = intercepto_canal + 0.5 * np.log(alvo / obs)
            intercepto_canal -= intercepto_canal[-1]          # fixa 'outros' como referência
    afinidade = _softmax(logits + intercepto_canal)           # (n, 5), soma 1 por linha

    # --- latente 3: sensibilidade a ser abordado ----------------------------------------------
    # Cliente que se incomoda de ser abordado. NÃO é feature do modelo — mas é fortemente
    # determinado por features que SÃO observáveis (idade alta, canal orgânico, zero engajamento
    # no app). Essa é a premissa que torna o problema resolvível: se a irritação fosse um
    # coin flip independente das features, nenhum modelo do mundo conseguiria antecipá-la, e o
    # braço nulo nunca seria escolhido para ninguém em particular.
    p_sensivel = _sigmoid(
        -3.0
        + 3.0 * (idade > 45)
        + 1.8 * organico
        + 1.6 * (sessoes_app_d0 == 0)
        + 0.9 * pago
    )
    sensivel_msg = rng.binomial(1, p_sensivel)

    latentes = {
        "p0": p0,
        "afinidade": afinidade,
        "sensivel_msg": sensivel_msg,
        "p_sensivel": p_sensivel,          # usado só para montar o benchmark "oráculo condicional"
        "intercepto_p0": intercepto_p0,
        "intercepto_canal": intercepto_canal,
    }
    return df, latentes


def prob_por_braco(latentes, usar_sensibilidade_esperada=False):
    """Matriz (n, n_bracos) com a probabilidade VERDADEIRA de ativar em 7 dias sob cada braço.

    Em produção isso é justamente o que não se conhece — é o que o bandit estima.

    `usar_sensibilidade_esperada=True` troca a realização 0/1 da irritação pela sua
    probabilidade. Serve para construir o benchmark honesto: nenhuma política consegue saber
    se ESTE cliente é dos que se irritam, só qual a chance disso. Ver a seção 7.
    """
    p0 = latentes["p0"]
    afinidade = latentes["afinidade"]
    sensivel = latentes["p_sensivel"] if usar_sensibilidade_esperada else latentes["sensivel_msg"]

    # Sino em p0: efeito máximo em quem está "em cima do muro" (p0 ~ 0.15), quase nulo em quem
    # nunca ativaria (p0 ~ 0) e em quem ativaria de qualquer jeito (p0 alto).
    persuadabilidade = np.exp(-0.5 * ((p0 - PICO_PERSUASAO) / LARGURA_PERSUASAO) ** 2)
    # Incomodar quem já ia ativar custa muito mais caro do que incomodar quem não ia.
    dano = DANO_MAX * sensivel * (0.3 + 2.0 * p0)

    p = np.empty((len(p0), len(BRACOS)))
    p[:, BRACO_NULO] = p0                         # braço nulo: nada muda, por definição
    for braco, canal in CANAL_DO_BRACO.items():
        ganho = GANHO_MAX * afinidade[:, CANAIS.index(canal)] * persuadabilidade
        p[:, braco] = p0 + ganho - dano
    return np.clip(p, 0.002, 0.98)


def simular_resposta(p_escolhida, rng):
    """Converte probabilidade em resultado observado: ativou (0/1) e em que dia (0..7)."""
    ativou = (rng.random(len(p_escolhida)) < p_escolhida).astype(int)
    # Dia da ativação: TAXA_ATIVACAO_D0 / TAXA_ATIVACAO_D7 dos ativadores caem no d0, o resto
    # se espalha pelos dias 1..7. Isso é o que obriga a esperar a janela antes de dar a recompensa.
    frac_d0 = TAXA_ATIVACAO_D0 / TAXA_ATIVACAO_D7
    peso_dias = np.array([frac_d0] + [(1 - frac_d0) / 7] * 7)
    dia = rng.choice(np.arange(JANELA_ATRIBUICAO_DIAS + 1), size=len(p_escolhida), p=peso_dias)
    return ativou, np.where(ativou == 1, dia, -1)


print("Gerador definido.")

In [ ]:
# ---------------------------------------------------------------------------------------------
# Histórico: ~26 semanas de clientes que abriram a conta ANTES do agente existir.
# Todos receberam, na prática, o braço nulo. É com isso que treinamos a propensão.
# ---------------------------------------------------------------------------------------------
SEMANAS_HISTORICO = 26
n_hist = SEMANAS_HISTORICO * CLIENTES_POR_SEMANA

rng = np.random.default_rng(SEED)
hist, lat_hist = gerar_populacao(n_hist, rng)

# Guardamos a calibração para reutilizar em todas as coortes futuras da simulação.
CALIB = {"intercepto_p0": lat_hist["intercepto_p0"], "intercepto_canal": lat_hist["intercepto_canal"]}

# Histórico = mundo sem agente ⇒ a resposta observada é a do braço nulo.
p_hist = prob_por_braco(lat_hist)
hist["ativou_d7"], hist["dia_ativacao"] = simular_resposta(p_hist[:, BRACO_NULO], rng)

# Canal de ativação (só existe para quem ativou) — sorteado pela afinidade latente.
# Sorteio multinomial vetorizado: compara um uniforme com a CDF acumulada de cada linha.
u = rng.random(n_hist)
cum = lat_hist["afinidade"].cumsum(axis=1)
canal_idx = (u[:, None] > cum).sum(axis=1).clip(0, 4)
hist["canal_ativacao"] = np.where(hist["ativou_d7"] == 1, np.array(CANAIS)[canal_idx], "nao_ativou")

print(f"Histórico: {len(hist):,} clientes ({SEMANAS_HISTORICO} semanas)\n")
print(f"Ativação d7 : {hist['ativou_d7'].mean():.2%}   (alvo {TAXA_ATIVACAO_D7:.0%})")
print(f"Ativação d0 : {((hist['dia_ativacao'] == 0)).mean():.2%}   (alvo {TAXA_ATIVACAO_D0:.0%})")
print("\nMix de canal entre os ativadores (alvo entre parênteses):")
mix_obs = hist.loc[hist["ativou_d7"] == 1, "canal_ativacao"].value_counts(normalize=True)
for c in CANAIS:
    print(f"  {c:<13} {mix_obs.get(c, 0):.2%}   ({MIX_CANAL_ATIVACAO[c]:.2%})")

hist.head()

## 2. Modelo de propensão

Alvo: `ativou_d7`. Features: só o que existe no momento da abertura da conta.

Duas exigências que vêm do fato de a saída virar **feature do bandit**:

1. **Calibração importa mais que ranking.** O bandit vai usar `p0` como número, não como
   ordenação — então olhamos Brier e a curva de calibração, não só AUC.
2. **O treino tem que ser livre de contaminação do agente.** Aqui é trivial (o histórico é
   todo pré-agente). Em produção, quando o agente estiver no ar, **retreine a propensão apenas
   com o grupo controle de 20%** — que continua sendo um mundo sem intervenção. Treinar com
   quem recebeu mensagem faz `p0` deixar de significar "ativaria sozinho" e o bandit passa a
   estimar uplift contra uma linha de base que se move.

In [ ]:
# ---------------------------------------------------------------------------------------------
# Treino da propensão
# ---------------------------------------------------------------------------------------------
FEATURES_NUM = [
    "idade", "renda_declarada", "score_bureau", "sessoes_app_d0",
    "ja_tem_conta_outro_banco", "cadastro_completo", "dispositivo_ios", "cartao_aprovado",
]
FEATURES_CAT = ["canal_aquisicao"]

X_hist = hist[FEATURES_NUM + FEATURES_CAT]
y_hist = hist["ativou_d7"].to_numpy()

# Split temporal seria o certo em produção (treinar no passado, validar no futuro).
# O gerador é estacionário, então um split aleatório estratificado basta aqui.
X_tr, X_te, y_tr, y_te = train_test_split(
    X_hist, y_hist, test_size=0.25, random_state=SEED, stratify=y_hist
)

preproc = ColumnTransformer([
    ("num", StandardScaler(), FEATURES_NUM),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), FEATURES_CAT),
])

# Regressão logística: naturalmente calibrada, rápida de servir, fácil de auditar.
# Para um problema deste tamanho ela é uma escolha defensável de produção.
modelo_propensao = Pipeline([
    ("preproc", preproc),
    ("clf", LogisticRegression(max_iter=1000, C=1.0)),
])
modelo_propensao.fit(X_tr, y_tr)

p_te = modelo_propensao.predict_proba(X_te)[:, 1]
print("Logistic Regression")
print(f"  AUC   : {roc_auc_score(y_te, p_te):.4f}")
print(f"  Brier : {brier_score_loss(y_te, p_te):.5f}")
print(f"  média prevista {p_te.mean():.4f} vs observada {y_te.mean():.4f}")

# Comparação com gradient boosting: se o ganho de AUC for pequeno, ficamos com a logística
# (mais simples de operar). Se for grande, vale trocar — mas aí calibre o boosting.
try:
    from xgboost import XGBClassifier
    modelo_gb = Pipeline([
        ("preproc", preproc),
        ("clf", XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
            eval_metric="logloss", random_state=SEED, n_jobs=-1,
        )),
    ])
    modelo_gb.fit(X_tr, y_tr)
    p_gb = modelo_gb.predict_proba(X_te)[:, 1]
    print("\nXGBoost")
    print(f"  AUC   : {roc_auc_score(y_te, p_gb):.4f}")
    print(f"  Brier : {brier_score_loss(y_te, p_gb):.5f}")
except ImportError:
    print("\n(xgboost não instalado — seguindo apenas com a logística)")

In [ ]:
# ---------------------------------------------------------------------------------------------
# Sanidade da propensão: calibração e poder de separação.
# A calibração é o gráfico que realmente importa aqui, porque p0 entra como feature no bandit.
# ---------------------------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

frac_pos, media_prev = calibration_curve(y_te, p_te, n_bins=20, strategy="quantile")
axes[0].plot([0, 0.6], [0, 0.6], "k--", lw=1, label="perfeito")
axes[0].plot(media_prev, frac_pos, "o-", label="modelo")
axes[0].set(xlabel="propensão prevista", ylabel="ativação observada", title="Calibração")
axes[0].legend()

axes[1].hist(p_te, bins=60, color="steelblue")
axes[1].axvline(y_te.mean(), color="crimson", ls="--", label=f"base {y_te.mean():.1%}")
axes[1].set(xlabel="propensão prevista", ylabel="clientes", title="Distribuição da propensão")
axes[1].legend()

# Ativação observada por decil de propensão: a "escada" que o time de negócio entende.
dec = pd.DataFrame({"p": p_te, "y": y_te})
dec["decil"] = pd.qcut(dec["p"], 10, labels=False) + 1
por_decil = dec.groupby("decil")["y"].mean()
axes[2].bar(por_decil.index, por_decil.to_numpy(), color="darkseagreen")
axes[2].axhline(y_te.mean(), color="crimson", ls="--", label=f"média {y_te.mean():.1%}")
axes[2].set(xlabel="decil de propensão", ylabel="ativação d7", title="Ativação por decil")
axes[2].legend()

plt.tight_layout()
plt.show()

print("Ativação: decil 1 = {:.1%} | decil 10 = {:.1%} | lift = {:.1f}x".format(
    por_decil.iloc[0], por_decil.iloc[-1], por_decil.iloc[-1] / por_decil.iloc[0]))

## 3. Contexto do bandit

O vetor de contexto é o mesmo para todos os braços (modelo **disjunto**: cada braço tem seu
próprio conjunto de coeficientes). Composição:

| Bloco | Por quê |
|---|---|
| `1` (intercepto) | nível base do braço |
| `p0`, `p0²` | deixa o efeito do braço variar em forma de sino com a propensão |
| `idade_z`, `idade_maior_45` | afinidade por seguro **e** proxy de irritação (efeito de limiar) |
| `sessoes_z`, `sem_sessao_d0` | engajamento: proxy de irritação (efeito de limiar) |
| `log_renda_z` | separa afinidade investimento vs pagamento |
| `canal_pago`, `canal_indicacao` | orgânico é a referência, e orgânico correlaciona com irritação |
| `ja_tem_conta_outro_banco`, `cartao_aprovado` | proxies de afinidade por pix / pagamento |

Três regras de ouro:

- **O contexto tem que conter os drivers do dano, na forma funcional certa.** O modelo é linear
  em `x`: se a irritação é um efeito de limiar (`idade > 45`, `sessões == 0`), a versão contínua
  `idade_z` não captura isso e o braço nulo nunca vai ganhar de ninguém. Foi por isso que
  `idade_maior_45` e `sem_sessao_d0` entraram — sem elas a política silencia quase ninguém.
- **A padronização precisa ser fixa**, calculada uma vez no histórico e congelada com o modelo.
  Se as médias/desvios mudarem entre treino e serving, os coeficientes aprendidos viram lixo.
- **Contexto enxuto.** São 12 dimensões × 5 braços = 60 parâmetros. Com ~23 mil impressões por
  semana isso converge rápido. Cada feature extra encarece a exploração.

In [ ]:
# ---------------------------------------------------------------------------------------------
# Construção do contexto (a mesma função é usada no treino, na simulação e no serving)
# ---------------------------------------------------------------------------------------------
NOMES_CONTEXTO = [
    "intercepto", "p0", "p0_quad",
    "idade_z", "idade_maior_45",          # idade_maior_45: o incômodo é um efeito de limiar
    "sessoes_z", "sem_sessao_d0",         # sem_sessao_d0: idem — zero sessões é qualitativamente diferente
    "log_renda_z",
    "canal_pago", "canal_indicacao",      # orgânico é a referência
    "ja_tem_conta_outro_banco", "cartao_aprovado",
]
DIM_CONTEXTO = len(NOMES_CONTEXTO)

# Estatísticas de padronização CONGELADAS no histórico. Vão junto com o artefato do modelo.
ESCALADOR = {
    "idade":     (float(hist["idade"].mean()), float(hist["idade"].std())),
    "sessoes":   (float(hist["sessoes_app_d0"].mean()), float(hist["sessoes_app_d0"].std())),
    "log_renda": (float(np.log(hist["renda_declarada"]).mean()), float(np.log(hist["renda_declarada"]).std())),
}


def construir_contexto(df, propensao):
    """(df de features, vetor de propensão) -> matriz de contexto (n, DIM_CONTEXTO)."""
    n = len(df)
    idade_m, idade_s = ESCALADOR["idade"]
    ses_m, ses_s = ESCALADOR["sessoes"]
    ren_m, ren_s = ESCALADOR["log_renda"]

    canal = df["canal_aquisicao"].to_numpy()
    idade = df["idade"].to_numpy()
    sessoes = df["sessoes_app_d0"].to_numpy()
    X = np.column_stack([
        np.ones(n),
        propensao,
        propensao ** 2,
        (idade - idade_m) / idade_s,
        (idade > 45).astype(float),
        (sessoes - ses_m) / ses_s,
        (sessoes == 0).astype(float),
        (np.log(df["renda_declarada"].to_numpy()) - ren_m) / ren_s,
        (canal == "pago").astype(float),
        (canal == "indicacao").astype(float),
        df["ja_tem_conta_outro_banco"].to_numpy().astype(float),
        df["cartao_aprovado"].to_numpy().astype(float),
    ])
    return X


def escorar_propensao(df):
    """Wrapper do modelo de propensão — em produção é uma chamada ao modelo publicado."""
    return modelo_propensao.predict_proba(df[FEATURES_NUM + FEATURES_CAT])[:, 1]


# Verificação rápida
_ctx = construir_contexto(hist.head(3), escorar_propensao(hist.head(3)))
pd.DataFrame(_ctx, columns=NOMES_CONTEXTO)

## 4. O MAB contextual: Linear Thompson Sampling

Um modelo bayesiano linear **por braço**, com recompensa binária (ativou em 7 dias):

```
A_a = λI + Σ x xᵀ          (informação acumulada pelo braço a)
b_a = Σ r x
θ̂_a = A_a⁻¹ b_a           (estimativa pontual dos coeficientes)
Cov  = v² A_a⁻¹            (incerteza sobre θ_a)
```

A cada decisão sorteamos `θ̃_a ~ N(θ̂_a, v² A_a⁻¹)` para cada braço e escolhemos
`argmax_a xᵀθ̃_a`. Braço com pouca evidência tem covariância grande, é sorteado longe da média
e por isso é testado — a exploração sai de graça, sem ε-greedy nem agenda de decaimento.

**Por que regressão linear com recompensa 0/1 e não bandit logístico?** Porque o update é
fechado (soma de matrizes), o estado é pequeno e serializável, o batch update é exato, e a
aproximação é ótima na prática nessa faixa de probabilidade. Logístico exigiria aproximação
variacional/Laplace a cada rodada — mais peças móveis para operar em troca de pouco ganho.

**`v` é o botão de exploração.** Maior = explora mais e mais tempo. Com recompensa binária de
média ~0.2, o desvio residual é ~0.4; `v = 0.25` explora com firmeza sem ser temerário.

**Regra que faz o braço nulo funcionar.** O braço nulo é atualizado exatamente como os outros:
quando ele é sorteado, não mandamos mensagem e observamos se o cliente ativou sozinho. Assim ele
aprende a linha de base **no mesmo espaço de contexto** dos demais braços, e a comparação
`xᵀθ_nulo` vs `xᵀθ_a` é uma estimativa direta de uplift.

In [ ]:
# ---------------------------------------------------------------------------------------------
# Implementação do LinTS
# ---------------------------------------------------------------------------------------------
class LinTS:
    """Linear Thompson Sampling com modelo disjunto por braço.

    Pronto para produção: estado pequeno, update em lote, serialização em JSON.
    """

    def __init__(self, n_bracos, dim, lambda_reg=1.0, v=0.25, seed=0):
        self.n_bracos = n_bracos
        self.dim = dim
        self.lambda_reg = lambda_reg
        self.v = v
        self.rng = np.random.default_rng(seed)
        # Um A e um b por braço. A começa em λI (prior de que todo coeficiente é ~0).
        self.A = np.array([np.eye(dim) * lambda_reg for _ in range(n_bracos)])
        self.b = np.zeros((n_bracos, dim))
        self.n_impressoes = np.zeros(n_bracos, dtype=int)
        self.n_recompensas = np.zeros(n_bracos)

    # --- estimativas ---------------------------------------------------------------------------
    def theta(self, braco):
        """Estimativa pontual (média a posteriori) dos coeficientes do braço."""
        return np.linalg.solve(self.A[braco], self.b[braco])

    def esperado(self, X):
        """Recompensa esperada (média a posteriori) de cada braço. Retorna (n, n_bracos).

        É o modo de serving determinístico — sem exploração.
        """
        return np.column_stack([X @ self.theta(a) for a in range(self.n_bracos)])

    def desvio(self, X):
        """Desvio-padrão a posteriori de xᵀθ por braço. Retorna (n, n_bracos).

        Usado para as guardrails de "não fazer dano" mais adiante.
        """
        out = np.empty((len(X), self.n_bracos))
        for a in range(self.n_bracos):
            A_inv = np.linalg.inv(self.A[a])
            # var = v² xᵀ A⁻¹ x, calculado linha a linha de forma vetorizada
            out[:, a] = self.v * np.sqrt(np.maximum(np.einsum("ij,jk,ik->i", X, A_inv, X), 0))
        return out

    # --- decisão ------------------------------------------------------------------------------
    def escolher(self, X):
        """Thompson Sampling: um θ sorteado POR CLIENTE e por braço. Retorna (braços, scores)."""
        n = len(X)
        scores = np.empty((n, self.n_bracos))
        for a in range(self.n_bracos):
            A_inv = np.linalg.inv(self.A[a])
            mu = A_inv @ self.b[a]
            # Cholesky da covariância v²A⁻¹ (+ jitter para garantir positividade numérica)
            L = np.linalg.cholesky((self.v ** 2) * A_inv + 1e-12 * np.eye(self.dim))
            ruido = self.rng.standard_normal((n, self.dim)) @ L.T
            theta_amostrado = mu[None, :] + ruido          # (n, dim): um θ por cliente
            scores[:, a] = np.einsum("ij,ij->i", X, theta_amostrado)
        return scores.argmax(axis=1), scores

    # --- aprendizado --------------------------------------------------------------------------
    def atualizar_lote(self, X, bracos, recompensas):
        """Update em lote — é assim que roda em produção, com a recompensa de 7 dias atrás."""
        for a in range(self.n_bracos):
            m = bracos == a
            if not m.any():
                continue
            Xa, ra = X[m], recompensas[m]
            self.A[a] += Xa.T @ Xa
            self.b[a] += Xa.T @ ra
            self.n_impressoes[a] += int(m.sum())
            self.n_recompensas[a] += float(ra.sum())

    # --- persistência -------------------------------------------------------------------------
    def to_dict(self):
        return {
            "n_bracos": self.n_bracos, "dim": self.dim,
            "lambda_reg": self.lambda_reg, "v": self.v,
            "A": self.A.tolist(), "b": self.b.tolist(),
            "n_impressoes": self.n_impressoes.tolist(),
            "n_recompensas": self.n_recompensas.tolist(),
            "nomes_contexto": NOMES_CONTEXTO, "nomes_bracos": BRACOS,
        }

    @classmethod
    def from_dict(cls, d, seed=0):
        obj = cls(d["n_bracos"], d["dim"], d["lambda_reg"], d["v"], seed=seed)
        obj.A = np.array(d["A"])
        obj.b = np.array(d["b"])
        obj.n_impressoes = np.array(d["n_impressoes"], dtype=int)
        obj.n_recompensas = np.array(d["n_recompensas"], dtype=float)
        return obj


print("LinTS pronto.")

## 5. Simulação: 16 semanas de operação

O laço abaixo é intencionalmente parecido com o cronograma real:

1. Chega a coorte da semana (~29 mil clientes).
2. **20% vão para o controle** e nunca entram no fluxo (recebem braço nulo, mas ficam fora do
   aprendizado do bandit — são a régua limpa do programa).
3. Os 80% restantes: escora propensão → monta contexto → TS escolhe o assunto → agente fala.
4. **A recompensa da coorte só fecha 7 dias depois.** Então o update do modelo na semana `s`
   usa a coorte da semana `s-1`. Isso significa que decidimos a semana toda com um modelo
   "atrasado" — e é bom ver que o bandit aprende bem assim mesmo.

Para termos de comparação, computamos em paralelo (usando o ground truth, o que só é possível
em simulação): controle, braço aleatório, melhor braço fixo global, e o oráculo personalizado.

In [ ]:
# ---------------------------------------------------------------------------------------------
# Loop de simulação
# ---------------------------------------------------------------------------------------------
N_SEMANAS = 16
rng_sim = np.random.default_rng(SEED + 1)

bandit = LinTS(n_bracos=len(BRACOS), dim=DIM_CONTEXTO, lambda_reg=1.0, v=0.25, seed=SEED)

pendente = None          # coorte da semana anterior, esperando fechar a janela de 7 dias
log_semanal = []
log_impressoes = []      # log por cliente (é o que vira tabela de dados em produção)

for semana in range(1, N_SEMANAS + 1):
    # (a) Fecha a janela da coorte anterior e aprende com ela ---------------------------------
    if pendente is not None:
        bandit.atualizar_lote(pendente["X"], pendente["bracos"], pendente["recompensa"])

    # (b) Nova coorte da semana ---------------------------------------------------------------
    df_c, lat_c = gerar_populacao(CLIENTES_POR_SEMANA, rng_sim, **CALIB)
    p_verdadeira = prob_por_braco(lat_c)                      # ground truth (só o simulador vê)
    # Versão com a irritação em valor esperado: define o teto ATINGÍVEL por qualquer política
    # (ver discussão na seção 7 — a irritação individual é um sorteio que nenhuma feature revela).
    p_condicional = prob_por_braco(lat_c, usar_sensibilidade_esperada=True)

    em_controle = rng_sim.random(len(df_c)) < FRACAO_CONTROLE
    idx_fluxo = np.where(~em_controle)[0]

    # (c) Decisão para quem está no fluxo ------------------------------------------------------
    df_f = df_c.iloc[idx_fluxo]
    p0_hat = escorar_propensao(df_f)                          # <-- propensão como feature
    X_f = construir_contexto(df_f, p0_hat)
    bracos_escolhidos, _ = bandit.escolher(X_f)

    # (d) O mundo responde --------------------------------------------------------------------
    p_escolhida = p_verdadeira[idx_fluxo, bracos_escolhidos]
    recompensa, _dia = simular_resposta(p_escolhida, rng_sim)

    # Guarda para o update da semana seguinte (a recompensa "só existe" depois de 7 dias)
    pendente = {"X": X_f, "bracos": bracos_escolhidos, "recompensa": recompensa.astype(float)}

    # (e) Grupo controle: sem agente ----------------------------------------------------------
    p_ctrl = p_verdadeira[em_controle, BRACO_NULO]
    recompensa_ctrl, _ = simular_resposta(p_ctrl, rng_sim)

    # (f) Baselines contrafactuais (média das probabilidades = menos ruído que realizar) -------
    p_todos_nulo = p_verdadeira[idx_fluxo, BRACO_NULO].mean()
    p_aleatorio = p_verdadeira[idx_fluxo][:, 1:].mean()             # sorteia entre as 4 mensagens
    p_melhor_fixo = p_verdadeira[idx_fluxo].mean(axis=0).max()      # melhor braço único global
    # Oráculo condicional: escolhe pelo valor esperado (o melhor possível), colhe no mundo real.
    braco_orac = p_condicional[idx_fluxo].argmax(axis=1)
    p_oraculo = p_verdadeira[idx_fluxo][np.arange(len(idx_fluxo)), braco_orac].mean()
    # Oráculo individual: teto teórico, inatingível. Só para referência.
    p_oraculo_ind = p_verdadeira[idx_fluxo].max(axis=1).mean()

    log_semanal.append({
        "semana": semana,
        "n_fluxo": len(idx_fluxo),
        "n_controle": int(em_controle.sum()),
        "bandit_realizado": recompensa.mean(),
        "bandit_esperado": p_escolhida.mean(),
        "controle_realizado": recompensa_ctrl.mean(),
        "sem_mensagem": p_todos_nulo,
        "aleatorio": p_aleatorio,
        "melhor_braco_fixo": p_melhor_fixo,
        "oraculo": p_oraculo,
        "oraculo_individual": p_oraculo_ind,
        "pct_braco_nulo": (bracos_escolhidos == BRACO_NULO).mean(),
        **{f"pct_{BRACOS[a]}": (bracos_escolhidos == a).mean() for a in range(len(BRACOS))},
    })
    log_impressoes.append(pd.DataFrame({
        "semana": semana,
        "propensao": p0_hat,
        "braco": np.array(BRACOS)[bracos_escolhidos],
        "braco_id": bracos_escolhidos,
        "ativou_d7": recompensa,
        "p_verdadeira_escolhida": p_escolhida,
        "p_verdadeira_nulo": p_verdadeira[idx_fluxo, BRACO_NULO],
        "melhor_p_verdadeira": p_verdadeira[idx_fluxo].max(axis=1),
        "nulo_e_otimo": p_verdadeira[idx_fluxo].argmax(axis=1) == BRACO_NULO,
        "sensivel_msg": lat_c["sensivel_msg"][idx_fluxo],
    }))

# Última coorte também entra no modelo (fecha a janela pendente)
bandit.atualizar_lote(pendente["X"], pendente["bracos"], pendente["recompensa"])

res = pd.DataFrame(log_semanal)
impr = pd.concat(log_impressoes, ignore_index=True)

print(f"Simulação: {N_SEMANAS} semanas, {impr.shape[0]:,} clientes no fluxo do agente\n")
res[["semana", "bandit_esperado", "sem_mensagem", "aleatorio", "melhor_braco_fixo",
     "oraculo", "pct_braco_nulo"]].round(4)

## 6. O bandit está aprendendo?

Referências para ler os gráficos:

- **`sem_mensagem`** — o que aconteceria se o agente não existisse. É o piso.
- **`aleatorio`** — mandar um assunto qualquer. Mostra quanto do ganho vem de *personalizar*
  em vez de simplesmente *falar*.
- **`melhor_braco_fixo`** — o melhor assunto único para toda a base, escolhido com informação
  perfeita. É o que um A/B test tradicional entregaria. **Bater isso** é o argumento para usar
  bandit contextual em vez de teste A/B.
- **`oraculo condicional`** — o teto atingível: melhor braço por cliente para quem conhece o
  modelo verdadeiro, mas não o sorteio individual de irritação. Ver a seção 7 para o porquê.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))

ax = axes[0]
ax.plot(res.semana, res.oraculo, "k--", lw=1.5, label="oráculo condicional (teto)")
ax.plot(res.semana, res.bandit_esperado, "o-", color="crimson", lw=2, label="bandit (LinTS)")
ax.plot(res.semana, res.melhor_braco_fixo, "s-", color="darkorange", label="melhor braço fixo")
ax.plot(res.semana, res.aleatorio, "^-", color="steelblue", label="braço aleatório")
ax.plot(res.semana, res.sem_mensagem, "-", color="gray", lw=2, label="sem agente (piso)")
ax.set(xlabel="semana", ylabel="ativação d7 esperada", title="Aprendizado semana a semana")
ax.legend(fontsize=8)

# Regret acumulado: quanto de ativação deixamos na mesa por não sermos o oráculo.
ax = axes[1]
regret_bandit = ((res.oraculo - res.bandit_esperado) * res.n_fluxo).cumsum()
regret_fixo = ((res.oraculo - res.melhor_braco_fixo) * res.n_fluxo).cumsum()
regret_alea = ((res.oraculo - res.aleatorio) * res.n_fluxo).cumsum()
ax.plot(res.semana, regret_bandit, "o-", color="crimson", label="bandit")
ax.plot(res.semana, regret_fixo, "s-", color="darkorange", label="melhor braço fixo")
ax.plot(res.semana, regret_alea, "^-", color="steelblue", label="aleatório")
ax.set(xlabel="semana", ylabel="ativações perdidas (acum.)", title="Regret acumulado")
ax.legend(fontsize=8)

# Mix de braços: aqui se vê a exploração inicial virando exploração dirigida.
ax = axes[2]
cols = [f"pct_{b}" for b in BRACOS]
ax.stackplot(res.semana, *[res[c] for c in cols], labels=BRACOS, alpha=0.85)
ax.set(xlabel="semana", ylabel="proporção", title="Mix de braços escolhidos", ylim=(0, 1))
ax.legend(fontsize=8, loc="lower right")

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------------------------
# Resultado consolidado. Usamos as últimas 4 semanas como "política em regime".
# ---------------------------------------------------------------------------------------------
regime = res.tail(4)
base = regime.sem_mensagem.mean()

linhas = []
for nome, valor in [
    ("Sem agente (piso)", regime.sem_mensagem.mean()),
    ("Braço aleatório", regime.aleatorio.mean()),
    ("Melhor braço fixo (A/B ideal)", regime.melhor_braco_fixo.mean()),
    ("Bandit contextual", regime.bandit_esperado.mean()),
    ("Oráculo condicional (teto atingível)", regime.oraculo.mean()),
    ("Oráculo individual (teto teórico)", regime.oraculo_individual.mean()),
]:
    linhas.append({
        "política": nome,
        "ativação d7": f"{valor:.2%}",
        "lift abs (p.p.)": f"{(valor - base) * 100:+.2f}",
        "lift rel": f"{(valor / base - 1) * 100:+.1f}%",
    })
print("Regime (últimas 4 semanas)\n")
print(pd.DataFrame(linhas).to_string(index=False))

ativacoes_ano = (regime.bandit_esperado.mean() - base) * CLIENTES_POR_SEMANA * (1 - FRACAO_CONTROLE) * 52
print(f"\nProjeção: ~{ativacoes_ano:,.0f} ativações incrementais/ano "
      f"(sobre os {1 - FRACAO_CONTROLE:.0%} da base que entram no fluxo)")

# Validação do controle: o realizado do controle deve bater com o contrafactual 'sem_mensagem'.
print(f"\nSanidade do controle: realizado {res.controle_realizado.mean():.3%} "
      f"vs contrafactual {res.sem_mensagem.mean():.3%}")

## 7. Onde o agente atrapalha: usando o braço nulo

É a pergunta do enunciado. O bandit responde comparando duas estimativas no mesmo contexto:

```
uplift_a(x) = xᵀθ_a − xᵀθ_nulo
```

Se `max_a uplift_a(x) < 0`, falar com esse cliente **reduz** a probabilidade de ativação: é um
*sleeping dog*. E como o modelo é bayesiano, temos também a incerteza dessa diferença:

```
var(uplift_a) = v² (xᵀA_a⁻¹x + xᵀA_nulo⁻¹x)     ⇒  P(uplift_a > 0) = Φ(μ / σ)
```

Isso permite uma política de serving **conservadora**: só falar quando houver confiança
razoável de que a mensagem ajuda. Comparamos as duas abaixo.

### Contra o que medir (isso é importante e é fácil de errar)

No simulador, uma parte dos clientes se irrita e outra não, e isso é um **sorteio individual**
que nenhuma feature revela. Existem então dois "certos" diferentes:

- **Oráculo individual** — sabe se *este* cliente vai se irritar. É inatingível por qualquer
  modelo, sempre, inclusive na vida real. Comparar a política com isso só produz um recall
  artificialmente baixo e nenhuma informação útil.
- **Oráculo condicional** — conhece a *probabilidade* de irritação dado o que é observável.
  É o **teto real** de qualquer política, e é contra ele que a política deve ser avaliada.

Reportamos os dois, mas precisão/recall são medidos contra o oráculo condicional.
Na prática o efeito disso é: a política não aponta "o cliente João vai se irritar", ela aponta
**segmentos em que o valor esperado de falar é negativo**. É essa a entrega do braço nulo.

In [ ]:
# ---------------------------------------------------------------------------------------------
# Escora uma população nova com o modelo treinado e classifica cada cliente.
# ---------------------------------------------------------------------------------------------
rng_av = np.random.default_rng(SEED + 99)
df_av, lat_av = gerar_populacao(60_000, rng_av, **CALIB)
p_av_verdadeira = prob_por_braco(lat_av)                                    # oráculo individual
p_av_condicional = prob_por_braco(lat_av, usar_sensibilidade_esperada=True)  # oráculo condicional

p0_av = escorar_propensao(df_av)
X_av = construir_contexto(df_av, p0_av)

mu = bandit.esperado(X_av)                       # (n, 5) recompensa esperada por braço
sd = bandit.desvio(X_av)                         # (n, 5) incerteza por braço

uplift = mu[:, 1:] - mu[:, [BRACO_NULO]]         # uplift estimado de cada mensagem
melhor_msg = uplift.argmax(axis=1) + 1           # id do melhor braço com mensagem
melhor_uplift = uplift.max(axis=1)

# Incerteza do uplift do melhor braço: var(a) + var(nulo) (braços independentes no modelo disjunto)
sd_uplift = np.sqrt(sd[np.arange(len(sd)), melhor_msg] ** 2 + sd[:, BRACO_NULO] ** 2)
prob_ajuda = norm.cdf(melhor_uplift / np.maximum(sd_uplift, 1e-9))

# --- Política A: argmax puro (fala se o uplift estimado for positivo) -------------------------
politica_argmax = np.where(melhor_uplift > 0, melhor_msg, BRACO_NULO)

# --- Política B: conservadora ("do no harm") --------------------------------------------------
# Só fala se houver >=70% de confiança de que a mensagem ajuda. Na dúvida, cala.
LIMIAR_CONFIANCA = 0.70
politica_conservadora = np.where(prob_ajuda >= LIMIAR_CONFIANCA, melhor_msg, BRACO_NULO)

# --- Verdade do simulador --------------------------------------------------------------------
# Oráculo individual (inatingível): sabe o resultado do sorteio de irritação de cada cliente.
nulo_e_otimo_individual = p_av_verdadeira.argmax(axis=1) == BRACO_NULO
# Oráculo condicional (teto real): sabe apenas a probabilidade de irritação dado o observável.
nulo_e_otimo = p_av_condicional.argmax(axis=1) == BRACO_NULO
# Política ótima condicional, usada como teto nas comparações de ativação.
braco_otimo_cond = p_av_condicional.argmax(axis=1)

piso = p_av_verdadeira[:, BRACO_NULO].mean()
teto = p_av_verdadeira[np.arange(len(braco_otimo_cond)), braco_otimo_cond].mean()

print(f"Oráculo individual  : em {nulo_e_otimo_individual.mean():.1%} dos clientes o melhor é não falar")
print(f"                      (inatingível — depende de um sorteio que nenhuma feature revela)")
print(f"Oráculo condicional : em {nulo_e_otimo.mean():.1%} dos clientes o valor ESPERADO de falar")
print(f"                      é negativo. Este é o alvo real da política.\n")

politicas = [
    ("argmax puro", politica_argmax),
    (f"conservadora (p>={LIMIAR_CONFIANCA})", politica_conservadora),
    ("sempre falar (melhor msg)", melhor_msg),
]
for nome, pol in politicas:
    silencia = pol == BRACO_NULO
    vp = (silencia & nulo_e_otimo).sum()
    fp = (silencia & ~nulo_e_otimo).sum()
    fn = (~silencia & nulo_e_otimo).sum()
    ativacao = p_av_verdadeira[np.arange(len(pol)), pol].mean()
    # % do gap entre piso (não fazer nada) e teto (oráculo condicional) que a política captura
    captura = (ativacao - piso) / (teto - piso)
    print(f"Política {nome}")
    print(f"  silencia          : {silencia.mean():.1%} dos clientes")
    print(f"  precisão / recall : {vp / max(vp + fp, 1):.1%} / {vp / max(vp + fn, 1):.1%}")
    print(f"  ativação d7       : {ativacao:.2%}  (captura {captura:.1%} do ganho disponível)")
print(f"\nReferências: piso (nunca falar) {piso:.2%} | teto (oráculo condicional) {teto:.2%}")

In [ ]:
# ---------------------------------------------------------------------------------------------
# Retrato dos clientes que o modelo decide silenciar (é o que o time de negócio vai querer ver)
# ---------------------------------------------------------------------------------------------
perfil = df_av.copy()
perfil["propensao"] = p0_av
perfil["silenciado"] = politica_conservadora == BRACO_NULO
perfil["uplift_estimado"] = melhor_uplift
perfil["sensivel_msg_real"] = lat_av["sensivel_msg"]     # latente, só para validar

resumo = perfil.groupby("silenciado").agg(
    clientes=("idade", "size"),
    idade_media=("idade", "mean"),
    propensao_media=("propensao", "mean"),
    sessoes_app_d0=("sessoes_app_d0", "mean"),
    pct_organico=("canal_aquisicao", lambda s: (s == "organico").mean()),
    pct_sensivel_real=("sensivel_msg_real", "mean"),
    uplift_estimado=("uplift_estimado", "mean"),
).round(3)
print("Perfil: silenciado (True) vs abordado (False)\n")
print(resumo.to_string())

fig, axes = plt.subplots(1, 3, figsize=(17, 4.4))

# (1) Uplift estimado vs uplift real por decil de propensão — a curva em sino que motivou o design.
d = pd.DataFrame({
    "p0": p0_av,
    "uplift_est": melhor_uplift,
    # uplift do oráculo CONDICIONAL — o teto honesto, ver discussão acima
    "uplift_real": p_av_condicional[:, 1:].max(axis=1) - p_av_condicional[:, BRACO_NULO],
})
d["decil"] = pd.qcut(d.p0, 10, labels=False) + 1
g = d.groupby("decil")[["uplift_est", "uplift_real"]].mean()
axes[0].plot(g.index, g.uplift_real, "o-", color="black", label="uplift real (oráculo cond.)")
axes[0].plot(g.index, g.uplift_est, "s--", color="crimson", label="uplift estimado (bandit)")
axes[0].axhline(0, color="gray", lw=1)
axes[0].set(xlabel="decil de propensão", ylabel="uplift do melhor assunto",
            title="Uplift por nível de propensão")
axes[0].legend(fontsize=8)

# (2) Taxa de silenciamento por decil de propensão
s = pd.DataFrame({"p0": p0_av, "sil": politica_conservadora == BRACO_NULO,
                  "sil_real": nulo_e_otimo})
s["decil"] = pd.qcut(s.p0, 10, labels=False) + 1
gs = s.groupby("decil")[["sil", "sil_real"]].mean()
axes[1].bar(gs.index - 0.2, gs.sil_real, width=0.4, label="deveria silenciar", color="black")
axes[1].bar(gs.index + 0.2, gs.sil, width=0.4, label="modelo silencia", color="crimson")
axes[1].set(xlabel="decil de propensão", ylabel="% silenciado",
            title="Onde o agente atrapalha")
axes[1].legend(fontsize=8)

# (3) Distribuição do uplift estimado — a massa à esquerda de 0 é a população sleeping dog.
axes[2].hist(melhor_uplift, bins=80, color="steelblue")
axes[2].axvline(0, color="crimson", ls="--", lw=2, label="uplift = 0")
axes[2].set(xlabel="uplift estimado do melhor assunto", ylabel="clientes",
            title="Distribuição do uplift estimado")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------------------------
# O que cada braço aprendeu. Coeficiente positivo em p0 num braço de mensagem = "esse assunto
# funciona melhor em quem já tem propensão alta".
# ---------------------------------------------------------------------------------------------
coefs = pd.DataFrame(
    {BRACOS[a]: bandit.theta(a) for a in range(len(BRACOS))}, index=NOMES_CONTEXTO
).round(4)
print("Coeficientes aprendidos (θ̂ por braço)\n")
print(coefs.to_string())

print("\nImpressões e ativação observada por braço durante a simulação\n")
print(pd.DataFrame({
    "braco": BRACOS,
    "impressoes": bandit.n_impressoes,
    "ativacoes": bandit.n_recompensas.astype(int),
    "taxa": (bandit.n_recompensas / np.maximum(bandit.n_impressoes, 1)).round(4),
}).to_string(index=False))

print("\nATENÇÃO ao ler a coluna 'taxa': ela NÃO é comparável entre braços. O bandit manda cada "
      "\nbraço para o público em que ele vai bem, então as taxas brutas estão confundidas por "
      "\nseleção. Para efeito causal, use o grupo controle de 20% ou o uplift do modelo.")

## 8. Serving: as três funções que vão para produção

O ciclo em produção tem duas metades desacopladas:

```
ONLINE  (cliente abre a conta)
  features → escorar_propensao → construir_contexto → decidir_assunto → agente conversa
           → grava impressão (com o contexto usado!)

OFFLINE (job diário)
  impressões de 7+ dias atrás  →  junta com ativação  →  bandit.atualizar_lote  →  publica estado
```

Duas coisas que **não** são detalhe:

- **Grave o vetor de contexto usado na decisão**, não só o `client_id`. Se você recalcular o
  contexto no dia do update, as features já mudaram (o cliente teve 7 dias de atividade) e o
  modelo aprende a associação errada. O contexto precisa ser o do momento da decisão.
- **Grave a versão do modelo de propensão.** Quando ela mudar, a escala de `p0` muda, e os
  coeficientes do bandit em `p0`/`p0²` passam a estar num espaço diferente.

In [ ]:
# ---------------------------------------------------------------------------------------------
# Função de decisão para servir online
# ---------------------------------------------------------------------------------------------
def decidir_assunto(df_clientes, bandit, modo="explorar", limiar_confianca=0.70):
    """Decide o assunto da conversa para uma lista de clientes recém-abertos.

    modo="explorar"    -> Thompson Sampling. É o modo padrão: continua aprendendo.
    modo="conservador" -> argmax da média a posteriori, e só fala se P(uplift>0) >= limiar.
                          Não aprende nada novo. Use para uma fatia da base se houver pressão
                          para "não incomodar cliente" enquanto o bandit ainda é jovem.

    Retorna DataFrame com a decisão e tudo que precisa ser logado.
    """
    p0 = escorar_propensao(df_clientes)
    X = construir_contexto(df_clientes, p0)

    if modo == "explorar":
        bracos, _ = bandit.escolher(X)
    elif modo == "conservador":
        mu, sd = bandit.esperado(X), bandit.desvio(X)
        up = mu[:, 1:] - mu[:, [BRACO_NULO]]
        melhor = up.argmax(axis=1) + 1
        sd_up = np.sqrt(sd[np.arange(len(sd)), melhor] ** 2 + sd[:, BRACO_NULO] ** 2)
        p_ajuda = norm.cdf(up.max(axis=1) / np.maximum(sd_up, 1e-9))
        bracos = np.where(p_ajuda >= limiar_confianca, melhor, BRACO_NULO)
    else:
        raise ValueError(f"modo desconhecido: {modo}")

    mu = bandit.esperado(X)
    return pd.DataFrame({
        "propensao": p0,
        "braco_id": bracos,
        "assunto": np.array(BRACOS)[bracos],
        "vai_falar": bracos != BRACO_NULO,
        "uplift_estimado": mu[np.arange(len(mu)), bracos] - mu[:, BRACO_NULO],
        # >>> logar o contexto é obrigatório para o update ficar correto <<<
        "contexto": [json.dumps(dict(zip(NOMES_CONTEXTO, np.round(row, 6).tolist()))) for row in X],
    })


# Demonstração nos dois modos. A amostra é montada de propósito com 5 clientes que a política
# quer abordar e 5 que ela quer silenciar, para dar para ver a diferença entre os modos.
idx_fala = np.where(politica_conservadora != BRACO_NULO)[0][:5]
idx_cala = np.where(politica_conservadora == BRACO_NULO)[0][:5]
novos = df_av.iloc[np.concatenate([idx_fala, idx_cala])]

print("Modo explorar (Thompson Sampling — padrão em produção):")
print(decidir_assunto(novos, bandit, modo="explorar")
      .drop(columns="contexto").round(4).to_string(index=False))
print("\nModo conservador (só fala com confiança >= 70%):")
print(decidir_assunto(novos, bandit, modo="conservador")
      .drop(columns="contexto").round(4).to_string(index=False))

In [ ]:
# ---------------------------------------------------------------------------------------------
# Job offline de update: pega impressões com janela fechada e aprende.
# ---------------------------------------------------------------------------------------------
def atualizar_do_log(bandit, df_log):
    """Atualiza o bandit a partir do log de impressões com a janela de 7 dias já fechada.

    df_log precisa ter: contexto (json), braco_id, ativou_d7 (0/1).
    Filtre a montante: apenas impressões com data_decisao <= hoje - 7 dias.
    """
    X = np.array([[json.loads(c)[k] for k in NOMES_CONTEXTO] for c in df_log["contexto"]])
    bandit.atualizar_lote(X, df_log["braco_id"].to_numpy(), df_log["ativou_d7"].to_numpy(float))
    return bandit


# Round-trip completo de persistência: salva estado, recarrega, confere que decide igual.
estado = bandit.to_dict()
caminho = "bandit_estado.json"
with open(caminho, "w") as f:
    json.dump(estado, f)

bandit_recarregado = LinTS.from_dict(json.load(open(caminho)), seed=SEED)
iguais = np.allclose(bandit.esperado(X_av[:500]), bandit_recarregado.esperado(X_av[:500]))
print(f"Estado salvo em '{caminho}' ({len(json.dumps(estado)) / 1024:.1f} KB)")
print(f"Recarregado produz as mesmas estimativas: {iguais}")

# Simulação do job offline com um lote novo
lote = decidir_assunto(df_av.iloc[:5_000], bandit_recarregado, modo="explorar")
lote["ativou_d7"] = simular_resposta(
    p_av_verdadeira[:5_000][np.arange(5_000), lote["braco_id"].to_numpy()], rng_av)[0]
antes = bandit_recarregado.n_impressoes.sum()
atualizar_do_log(bandit_recarregado, lote)
print(f"\nJob offline: impressões acumuladas {antes:,} -> {bandit_recarregado.n_impressoes.sum():,}")

## 9. Checklist para colocar no ar

### Cronograma de dados (a parte que mais dá errado)

| Quando | O que roda |
|---|---|
| D0, cliente abre a conta | escora propensão, decide assunto, **loga contexto + braço + versão dos modelos** |
| D0–D7 | agente conversa; nenhuma recompensa existe ainda |
| D+8, job diário | pega impressões de 8 dias atrás, junta ativação, `atualizar_lote`, publica novo estado |

Rodar o update diariamente (em lote) e não semanalmente é melhor: o atraso efetivo do
aprendizado cai de ~14 para ~8 dias, com o mesmo código.

### Esquema mínimo da tabela de impressões

`client_id`, `data_decisao`, `versao_modelo_propensao`, `versao_politica`, `propensao`,
`contexto_json`, `braco_id`, `modo_decisao` (explorar/conservador), `grupo` (fluxo/controle),
`ativou_d7`, `dia_ativacao`, `canal_ativacao`.

### Rollout

1. **Shadow (1–2 semanas).** Política decide e loga, agente não fala. Valida pipeline e o
   cronograma dos 7 dias sem risco nenhum.
2. **Burn-in.** Abra para 100% do fluxo (os 80% fora do controle) com TS puro. Aceite que as
   primeiras semanas rendem pouco: é o preço da exploração, e é o que gera os dados de uplift
   que ninguém tem hoje. Se houver restrição de "não incomodar", limite a exploração servindo
   parte da base em modo conservador — mas saiba que essa fatia não gera aprendizado.
3. **Regime.** TS segue rodando: ele reexplora sozinho quando a população muda.

### Monitoramento

- **Ativação fluxo vs controle** — é o KPI do programa. Único número causal limpo.
- **Mix de braços por semana** — colapsar num braço só cedo demais indica `v` baixo ou bug de
  contexto. O `%` do braço nulo é a métrica de "quanto estamos protegendo cliente".
- **Distribuição da propensão** — drift aqui invalida os coeficientes em `p0`/`p0²`.
- **Calibração da propensão medida no controle** — a única fatia onde ela ainda é medível.
- **Cobertura de recompensa** — % de impressões que fecharam a janela e viraram update. Se cair,
  o bandit está aprendendo com dados truncados e vai subestimar todo mundo.

### Cuidados

- **Não retreine a propensão com quem recebeu mensagem** — use só o controle de 20%.
- **Ao publicar propensão nova, considere resetar (ou amortecer) o bandit** nas dimensões
  `p0`/`p0²`: a feature mudou de escala.
- **`v` é o botão de risco.** Suba para explorar mais, desça para consolidar. Mudanças aqui
  são reversíveis e não exigem retreino.
- **Braços novos** (as outras formas de ativação, quando entrarem) começam com `A = λI`, sem
  histórico — o TS vai explorá-los naturalmente. Não precisa recomeçar nada.
- **Frequência.** Este notebook modela uma decisão por cliente. Se o agente for falar mais de
  uma vez na janela, a unidade de decisão passa a ser (cliente, contato) e o contexto deve
  incluir quantas mensagens já foram enviadas — senão o bandit não consegue aprender fadiga.